In [10]:
"""
attention ->it was like on which parts does the input should focus on

"The cat sat on the mat"
when processing the word cat,it have the most attention to the cat,because cat (who sat)
medium attention -> mat (where)
less attention -> the

this is the core part of the whole attention

firstly
there is a book on the table
so this is an example 
so the Q and K relationship is a matrix
where the shape is (T,T)
we can see how does word gives attention to the other word in that matrix
so we can visualize them in the

          there    is      a      book 
there     ....     ....   ...     .....
is        ....     ....   ...     .....
a         ....     ....   ...     .....
book      ....     ....   ...     .....


these dots represent the scores of how much one word gives attention to another
as i don't know either 😂️ so it's better to take dots .....

we can see this attention and then
the query may ask in this way,take this as an example only,i don't know how exactly it might ask 😂️
Q -> i need a book (usually they aren't in this plain english,but to understand for a bit,we can assume just to make sure,to know,what exactly it was fetching for ?)
as all the keys are gonna say something right,so it would be more attetntion score to the last key book compared to those others and then that last one have the high attention than these others
so in this way they will reresent the Q and K relationship
and then are we done? not yet
because to this part we just did discovered which have the more attention to the each query
and then,what we need ?? we need the value right

so once the moment the scores are found,we reach out for the values section where this
whole matrix is multiplied by the values ,so this is what we do 
(Q @ K.T/sqrt(d_k)) @ V right
so dont need to think of the terminology for now
we can come back later for them
so this is what the attention is 

 
to this point we know the Q K V 
and we need to  think into the inside of this matrix
so how do we find the scores between these
we can see these values right before us,and when we go inside them
we will be introduces to the three 
weights -> w_q w_k w_v
so what exactly are these,and we are just one level above the basement ,which we have the tokens only
so this is the next layer to that basement,as we know about the token right?
and then,how does they contribute to the scores? i mean,yeah we know token did exist but at what kind of attention do they provide each other is the layer 3 and then
we are that at second layer which actually brings us to that layer 3
and ,this is where the weights are introduces as they are already trained on the datatset and then they can find the meaningful projections ,at first they are all random and the moment they are trained and they are made into the better ways of the forming the queries and then keys and then values
we have already taken one example right ,there is a book under the table
and,what if the sentence is changed and then how can we project them now
so these weights are trained in a way that,they are the secret extra ingredients to the food we are baking
so,when the inputs are multiplied as these weights are trained ,so now these inputs
can be formed to ask the queries and then keys and ,even values so that
now we have the clear map of the how do the values are represented
because we need to generate the text too right,not in this step but will soon introduce it too
and then ,with those we are converting the base tokens to the meaningful representations 
and then now we get the Q K and then we will use the softmax and then make them sum upto to the 1 and then we will multiply with the V to get the values 

Why do we need multiple heads?

Imagine the sentence has several relationships at once. One attention mechanism might learn to focus on one kind of relationship
while another can learn a different one. Multi-head attention gives the model several attention mechanisms operating in parallel.

    
in the multi head attention,we can take the multiple heads and then each head works on its own and then at the end we mix all of what they did find out

this is the order i did followed ,and the reaons are below
so, the first parameter is the num_dims because we need to say in how many dimensions we are gonna projecting these weights
and then next we need to give the number of head so that they will split into match the number of dimensions right

as the head_dims is like
in the entire dimensions ,how many dimensions does a single head gets
its like dividing the items into the equal number of pieces for all heads
as multiple heads can project over different and then we can get more information

now the Q K V can be attained by multiplying the weights with the x 
as we turn off the bias,we don't actually need them here,because it was like,just adds some threshold to the scores and that doesn't make anything diff,so its redundant here so we just tru to find the linear layer calculations without the bias
and once we get the Q K V
the next step is,now the actual shape of the Q -> (B,T,D)
as the B specifies the Batch,gpu can handle multiple batches so we use the batches of the tokens
as that B represents the batches
and the T represents the tokens which we have taken an example right
and the D is the number of dimensions
as we have splitted them into the heads and head_dims
we need to view that Q as the B T num_heads head_dim and then we need to transpose this Q matrix because ,we don't perform any kind of the operation on the head we perfom on the tokens and then dimensions of the heads
we make transpose the dimensions of the num_heads to the Tokens
and now it would looks like
B num_heads T head_dims
if you observe this is,number of batches -> out of number of batches -> what are the number of heads that performs parallel -> and inside that each heads we perform the matrix multiplication 
and we got the new Q and K (we just make them view in the way we wanted so i was referring this as new,not any other new matrix which comes from air 😂️)

and we can now check the shape of the
Q - > (B,num_heads,T,head_dims)
K - > (B,num_heads,T,head_dims)
so how does this matrix multiplications works ? 
so we need to transpose the dimensions of the K
so we are tranpose those last two dimensions of the K

and then we do the multiplication -> (T,head_dims) @ (head_dim,T) => (T,T)
and this is how we end up with the (T,T) which we have seen earlier 

so we are at a phase of the intermediate
and then once the scores are done we need to mask it right
because in training it can see the next words but in the inference we can't let the model to see netx tokens which are in future so we will make them to the -inf
and then after that we can use the softmax so the e^-inf will goes to the 1/e^inf and then it would be 0,so softmax can make it 0 
then we can calculate the ones with the probabilities and then after this we can strat multiplying wiht the Values
and then we would get the attention out

softmax is the function which works in this way  ==> e^x/sum(e^x) so it was like same probability,like individual values by the sum of the all values but we are using this e^x function

in the prev steps we have make the transpose right and we will now revert it and then make this blocks contiguous and then view these as the B T D
because we need to convert them back to the shape we did actually taken them from

and now it is the all the heads are sitting side by side with their own features
like B T D as the total dims are the 512 (for ex...) and then we can take the heads as the 8 right
so [0....63][64.....127][]....[.....511] so these are sitting side by side without any mixing of the info which all have gathered and then
now we use the final projections to make this clear

out -> shape(B,T,D)

proj_out -> shape(B,D,D)

multiplication goes in this way
(T,D) @ (D,D)
now taking a single row
for the first token
0 1 2 3 4 5 6 ............512 this is for the first token and this multiplies with the @ cols of that first proj matrix 
so each of the 512 features are gonna multiplied to the proj_matrix and then comes to the single features by addig what they all gathered 
and then we will get same 512 features again

so this final piece is to integrate the all the pieces which have worked independently
as these are all started random and then ,they all are trained and then we can get the correct matrix weights so they can give the predictions and then find the loss and then adjusts the values
so after many training steps they will become better and then they mill merge better

Attention(Q, K, V) = softmax(Q @ K.T / sqrt(d_k)) @ V

Q - > query (what am i looking for ??)
K - > key   (what do i provide..)
V - > value (what information i gave)

Q @ K.T -> similarity these are the scores
softmax -> it convert the scores to the probabilities

/sqrt(d_k) - > to prevent the softmax saturation [d_k is the dimensions of the head]

"""


import torch
import torch.nn as nn
import math
import torch.nn.functional as F


class CasualSelfAttention(nn.Module):
    def __init__(self,num_dims,num_heads):
        super().__init__()
        self.num_dims = num_dims
        self.num_heads = num_heads

        assert self.num_dims % self.num_heads == 0 , "enter correct number of heads"
        # i know you might haven't done any mistake,in any case ,just to make sure it won't go wrong

        self.head_dims = num_dims // num_heads

        self.w_q = nn.Linear(num_dims,num_dims,bias=False)
        self.w_k = nn.Linear(num_dims,num_dims,bias=False)
        self.w_v = nn.Linear(num_dims,num_dims,bias=False)

        self.proj_out = nn.Linear(num_dims,num_dims)

    def forward(self,x):
        B,T,D = x.shape

        Q = self.w_q(x)
        K = self.w_k(x)
        V = self.w_v(x)

        Q = Q.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        K = K.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        V = V.view(B,T,self.num_heads,self.head_dims).transpose(1,2)

        scores = Q @ K.transpose(-2,-1) / math.sqrt(self.head_dims)

        """
        The argument diagonal controls which diagonal to consider.
        If diagonal = 0, all elements on and above the main diagonal are retained. A positive value excludes just as many diagonals above the main diagonal
        and similarly a negative value includes just as many diagonals below the main diagonal.
        The main diagonal are the set of indices {(i,i)}{(i,i)} for i∈[0,min⁡{d1,d2}−1]i∈[0,min{d1​,d2​}−1] where d1,d2d1​,d2​ are the dimensions of the matrix.
        
        """
        mask = torch.triu(torch.ones(T,T,dtype=torch.bool),diagonal=1) #so the elements below the diagonal becomes zero,we need this because we are using these to mask the inf

        # Tensor.masked_fill(mask, value) → Tensor
        scores = torch.masked_fill(scores,mask,float("-inf"))

        attention_out = F.softmax(scores,dim=-1)

        # multiply with V
        attention_out = attention_out @ V

        attention_out = attention_out.transpose(1,2).contiguous().view(B,T,D)   

        out = self.proj_out(attention_out)

        return out


if __name__=="__main__":
    num_dims=512
    num_heads=8

    attention = CasualSelfAttention(num_dims, num_heads)

    batch_size = 4
    seq_len = 10
    x = torch.randn(batch_size, seq_len, num_dims)

    output = attention.forward(x)
    
    print(f"Input shape: {x.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Expected shape: [{batch_size}, {seq_len}, {num_dims}]")
        
                 

Input shape: torch.Size([4, 10, 512])
Output shape: torch.Size([4, 10, 512])
Expected shape: [4, 10, 512]
